---
title: "Від чого залежить ціна діаманта: регресійне моделювання"
subtitle: "Лабораторна робота №3. Основи регресійного моделювання"
author: "Демчук Владислав"
date: last-modified
lang: uk
format:
  html:
    toc: true
    toc-depth: 2
    number-sections: true
    code-fold: true
    code-summary: "Код"
    embed-resources: true
    df-print: kable
    fig-dpi: 120
bibliography: references.bib
nocite: |
  @james2023, @nist, @kuhn2022, @sandve2013
execute:
  warning: false
  message: false
jupyter: python3
---

In [ ]:
#| label: setup
#| code-summary: "Пакети й налаштування"
import hashlib, json, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import OLSInfluence
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import (OneHotEncoder, OrdinalEncoder, FunctionTransformer,
                                   SplineTransformer, PolynomialFeatures)
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from IPython.display import Markdown

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

def savefig(name):
    plt.savefig(f"figures/{name}.png", dpi=150, bbox_inches="tight")

def rnd(x, n=None):
    return round(float(x)) if n is None else round(float(x), n)

def usd(v):
    return f"{v:,.0f}".replace(",", " ") + " $"

KEY = {}   # ключові числа для презентації

# Вступ

## Предметна область, ціль і питання

Ціна діаманта на роздрібному ринку визначається так званими «4C»: вагою (*carat*), огранкою (*cut*),
кольором (*color*) і чистотою (*clarity*). Продовжую лабораторну №2 на тому самому наборі **Diamonds**.

- **Одиниця спостереження** — один огранений круглий діамант з каталогу онлайн-продавця.
- **Ціль** — `price`, ціна в доларах США (ціни 2008 року), числова й неперервна.
- **Дослідницьке питання.** Наскільки точно чотири характеристики з сертифіката (4C) прогнозують ціну
  *нового* діаманта з того самого ринку і як ціна пов'язана з вагою, якщо огранка, колір і чистота однакові?
- **Для яких спостережень потрібен прогноз.** Для круглих діамантів того самого типу й періоду,
  у межах діапазону даних (0.2–5 ct, ціна до 18 823 $). Наприклад, покупець чи продавець хоче оцінити,
  чи справедлива ціна каменя з відомими 4C.

## Що підказує література

1. **Chu [-@chu2001]** будує модель ціни діамантів за 4C на даних сінгапурських оголошень. Ціна зростає
   з вагою швидше, ніж пропорційно, а якісні характеристики працюють як надбавки до ціни.
   Звідси **вибір змінних** (4C як основні предиктори) і **форма моделі**: логарифм ціни дає
   мультиплікативні (відсоткові) ефекти колір/чистота і рівномірніший розкид залишків.
2. **Scott і Yelowitz [-@scott2010]** на даних трьох великих онлайн-продавців показали **стрибки ціни на
   «круглих» вагах**: камінь 0.5 ct дорожчий за 0.49 ct на 18 % і більше, а 1 ct дорожчий за трохи легший
   на 5–10 %. Це підказує **межі інтерпретації**: зв'язок ціни з вагою не гладкий, тому проста
   степенева модель систематично помилятиметься біля порогів 0.5, 1, 1.5, 2 ct. Звідси моя модифікація
   регресії (порогові змінні і сплайн).
3. **Wickham і Grolemund [-@r4ds]** на цьому самому наборі показують, що в шкалі log–log зв'язок ціни з
   вагою майже лінійний і що «гірша» якість здається дорожчою через змішування з вагою. Це методологічна
   підстава завжди оцінювати ефекти якості **за сталої ваги**.

## Гіпотези (записані до оцінювання моделей)

- **H1 (вага, з EDA і літератури).** За однакових огранки, кольору й чистоти ціна зростає з вагою
  **швидше, ніж пропорційно**: еластичність ціни за вагою $\beta_{\log carat} > 1$.
  *Перевірка:* знак, величина й 95 % довірчий інтервал коефіцієнта при $\log(carat)$ у заздалегідь
  описаній моделі M2 (нижче), стійкі до гетероскедастичності стандартні похибки HC3.
- **H2 (чистота, з EDA і літератури).** За однакової ваги, огранки й кольору ціна **монотонно зростає**
  з чистотою від I1 до IF, і надбавка IF порівняно з I1 перевищує 100 %.
  *Перевірка:* коефіцієнти рівнів `clarity` у M2: знаки, порядок і 95 % ДІ надбавки $e^{\beta}-1$.

::: {.callout-warning}
Обидві гіпотези **виникли з EDA** лабораторної №2 (нахил 1.68 у log–log; чистота найсильніше
пов'язана з ціною каменів 1 ct), яке охоплювало **весь** набір. Тому їх перевірка на тих самих даних
є **розвідувальною**, а не підтверджувальною.
:::

# Методи

## Дані

In [ ]:
#| label: data
RAW = "data/raw/diamonds.csv"
URL = "https://raw.githubusercontent.com/tidyverse/ggplot2/main/data-raw/diamonds.csv"
if not os.path.exists(RAW):
    os.makedirs("data/raw", exist_ok=True)
    pd.read_csv(URL).to_csv(RAW, index=False)

sha = hashlib.sha256(open(RAW, "rb").read()).hexdigest()
raw = pd.read_csv(RAW)

# ті самі правила очищення, що й у лабораторній №2 (правила для окремих рядків,
# нічого не «вчать» з даних, тому їх можна застосувати до поділу)
df = raw.drop_duplicates()
df = df[(df.x > 0) & (df.y > 0) & (df.z > 0)]
depth_calc = df.z / ((df.x + df.y) / 2) * 100
df = df[(depth_calc - df.depth).abs() <= 5].reset_index(drop=True)

CATS = {"cut": ["Fair", "Good", "Very Good", "Premium", "Ideal"],
        "color": ["J", "I", "H", "G", "F", "E", "D"],                 # від гіршого до кращого
        "clarity": ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]}
for c, lv in CATS.items():
    df[c] = pd.Categorical(df[c], categories=lv, ordered=True)
df["log_price"] = np.log(df.price)
df["log_carat"] = np.log(df.carat)
KEY.update(n_raw=len(raw), n_clean=len(df), sha=sha)
print(f"Рядків у файлі: {len(raw)}, після очищення: {len(df)}; SHA-256: {sha[:16]}…")

- **Джерело:** набір `diamonds` з пакета ggplot2 [@ggplot2], файл `data-raw/diamonds.csv` з репозиторію
  tidyverse/ggplot2 (гілка `main`), отримано **листопад 2026**. Контрольна сума SHA-256 файлу
  `{python} sha[:16]`… записана в звіті, тож ту саму версію можна перевірити.
- **Умови використання:** ліцензія MIT пакета ggplot2; дані відкриті.
- **Розмір:** `{python} f"{len(raw):,}".replace(",", " ")` рядків і 10 змінних; після очищення
  `{python} f"{len(df):,}".replace(",", " ")` (вилучено повні дублікати, камені з нульовими розмірами і
  рядки, де `depth` суперечить розмірам x, y, z).
- **Обмеження даних:** спостережні дані одного ринку 2008 року; **немає дати й продавця**; ціни обрізані
  зверху (максимум 18 823 $); у наборі тільки круглі камені. Модель не можна переносити на інші роки,
  форми чи синтетичні діаманти.

## Поділ і захист від витоку

Спостереження незалежні (один рядок — окремий камінь, повторних вимірювань і часу немає), тому
**випадковий поділ 80/20 з `seed = 42`**. Поділ зафіксовано **до** підбору моделей; тестова частина
не використовується аж до фінальної оцінки.

In [ ]:
#| label: split
train, test = train_test_split(df, test_size=0.2, random_state=SEED)
FEATS = ["carat", "cut", "color", "clarity"]
X_tr, y_tr = train[FEATS], train.price
X_te, y_te = test[FEATS], test.price
pd.DataFrame({"частина": ["навчальна", "тестова"], "рядків": [len(train), len(test)],
              "середня ціна, $": [rnd(y_tr.mean()), rnd(y_te.mean())]})

- Усі графіки розділу 2.3 і вся діагностика побудовані **тільки на навчальній частині**.
- **Обмеження незалежності тесту.** Лабораторна №2 бачила весь набір, зокрема майбутню тестову частину.
  Вона вже підказала рішення: логарифмувати ціну й вагу, відкинути x, y, z (дублюють вагу), не брати
  `depth` і `table` (слабкий зв'язок), а також гіпотези H1–H2. Тому тестова оцінка трохи оптимістична.
  Нових даних з того самого ринку немає, отже повністю незалежної перевірки зробити не можу.
- **Доступність предикторів.** 4C вказані в сертифікаті до продажу, тож вони відомі на момент прогнозу.
  Ціну ніде не використано як предиктор.
- **Перехресна валідація:** 5-кратна `KFold(shuffle=True, random_state=42)` на навчальній частині,
  **однакові розбиття для всіх моделей**. Кодування категорій, логарифмування, сплайни й пороги
  входять у `Pipeline`, тож на кожному розбитті вони будуються лише за його навчальною частиною.
  Пропусків немає, тож імпутація не потрібна; масштабування лінійній регресії без штрафу не потрібне.

## Що з EDA переношу в моделювання

In [ ]:
#| label: fig-eda
#| fig-cap: "EDA на навчальній частині. Ліворуч: ціна має сильну праву асиметрію, після логарифма розподіл майже симетричний (кілька піків — популярні ваги каменів). Праворуч: у шкалі log–log ціна майже лінійно залежить від ваги, але видно вертикальні «сходинки» на 0.5, 1, 1.5 і 2 ct."
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
sns.histplot(train.price, bins=60, ax=ax[0], color="steelblue")
ax[0].set(title="Ціна", xlabel="Ціна, $", ylabel="Кількість діамантів")
sns.histplot(train.log_price, bins=60, ax=ax[1], color="seagreen")
ax[1].set(title="Логарифм ціни", xlabel="ln(ціна, $)", ylabel="")
ax[2].scatter(train.carat, train.price, s=2, alpha=0.08)
ax[2].set(xscale="log", yscale="log", title="Ціна і вага (log–log)",
          xlabel="Вага, ct (log)", ylabel="Ціна, $ (log)")
for t in (0.5, 1, 1.5, 2):
    ax[2].axvline(t, color="red", lw=0.8, ls="--")
ax[2].minorticks_off()
ax[2].set_xticks([0.2, 0.3, 0.5, 1, 2, 3, 5], ["0.2", "0.3", "0.5", "1", "2", "3", "5"])
ax[2].set_yticks([500, 1000, 2000, 5000, 10000], ["500", "1000", "2000", "5000", "10000"])
plt.tight_layout(); savefig("01_eda_target"); plt.show()
KEY["skew_price"] = rnd(train.price.skew(), 2)
KEY["skew_log"] = rnd(train.log_price.skew(), 2)

In [ ]:
#| label: fig-eda-clarity
#| fig-cap: "Медіанна ціна за чистотою: без контролю ваги (ліворуч) найчистіші камені здаються дешевшими; серед каменів 0.9–1.1 ct (праворуч) ціна чітко зростає з чистотою. Отже, ефекти якості треба оцінювати за сталої ваги."
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
m_all = train.groupby("clarity", observed=True).price.median()
m_1ct = train[train.carat.between(0.9, 1.1)].groupby("clarity", observed=True).price.median()
ax[0].bar(m_all.index.astype(str), m_all.values, color="lightgray")
ax[0].set(title="Усі камені", xlabel="Чистота (від гіршої до кращої)", ylabel="Медіанна ціна, $")
ax[1].bar(m_1ct.index.astype(str), m_1ct.values, color="steelblue")
ax[1].set(title="Камені 0.9–1.1 ct", xlabel="Чистота (від гіршої до кращої)", ylabel="")
plt.tight_layout(); savefig("02_eda_clarity"); plt.show()

Найважливіші висновки з лабораторної №2, перевірені тут на навчальній частині:

- **Розподіл цілі:** асиметрія ціни `{python} KEY["skew_price"]`, логарифма ціни `{python} KEY["skew_log"]`.
  *Рішення:* модель для $\ln(price)$; ефекти тоді у відсотках, а розкид залишків рівніший.
- **Пропуски:** немає. **Викиди:** дорогі й великі камені реальні, їх не видаляю; помилкові рядки
  вже вилучено на очищенні.
- **Зв'язки й нелінійність:** зв'язок ціни з вагою степеневий (у log–log майже пряма) зі «сходинками»
  на круглих вагах. *Рішення:* $\ln(carat)$ у базовій лінійній моделі; сплайн і пороги в модифікації.
- **Змішування:** «краща» якість частіше в дрібних каменях (@fig-eda-clarity). *Рішення:* завжди
  тримати вагу в моделі, коли інтерпретуємо огранку, колір і чистоту.
- **Дублювання:** x, y, z майже ідеально корелюють з вагою (r ≈ 0.98), тому не входять у модель.

## Метрики й правило вибору (зафіксовано до відкриття тесту)

- **Основна метрика — MAE**, у доларах: «на скільки в середньому помиляється оцінка ціни». Розподіл ціни
  дуже асиметричний, і RMSE домінували б кілька найдорожчих каменів. RMSE і $R^2$ подаю як додаткові.
- **Правило вибору:** модель з найменшим середнім MAE на 5 розбиттях. Якщо простіша модель гірша менше
  ніж на одне стандартне відхилення MAE за розбиттями, беру простішу.
- На тесті один раз оцінюю обрану модель і константний орієнтир. Найкращу **лінійну** модель (вона
  потрібна для інтерпретації гіпотез) теж один раз оцінюю на тесті, це рішення прийняте заздалегідь.
- Для моделей на $\ln(price)$ прогноз у доларах — $e^{\hat y}$, тобто оцінка **медіани** ціни.
  Вона узгоджується з MAE, але трохи занижує середню ціну [@duan1983].

# Результати

In [ ]:
#| label: models
#| code-summary: "Опис усіх моделей (однаковий протокол)"
def onehot():
    return OneHotEncoder(categories=[CATS[c] for c in ["cut", "color", "clarity"]], drop="first")

def thresholds(c):
    c = np.asarray(c).ravel()
    return np.column_stack([c >= t for t in (0.5, 1.0, 1.5, 2.0)]).astype(float)

QUAL = ["cut", "color", "clarity"]
log_target = lambda est: TransformedTargetRegressor(est, func=np.log, inverse_func=np.exp)

MODELS = {
    "M0 константа (середнє)": DummyRegressor(strategy="mean"),
    "M1 лінійна, ціна в $": make_pipeline(
        ColumnTransformer([("w", "passthrough", ["carat"]), ("q", onehot(), QUAL)]),
        LinearRegression()),
    "M2 лінійна log–log": log_target(make_pipeline(
        ColumnTransformer([("w", FunctionTransformer(np.log), ["carat"]), ("q", onehot(), QUAL)]),
        LinearRegression())),
    "M3 log–log + сплайн і пороги ваги": log_target(make_pipeline(
        ColumnTransformer([
            ("s", make_pipeline(FunctionTransformer(np.log), SplineTransformer(n_knots=6, degree=3)), ["carat"]),
            ("t", FunctionTransformer(thresholds), ["carat"]),
            ("q", onehot(), QUAL)]),
        LinearRegression())),
    "M4 = M3 + взаємодії вага×колір, вага×чистота": log_target(make_pipeline(
        ColumnTransformer([
            ("s", make_pipeline(FunctionTransformer(np.log), SplineTransformer(n_knots=6, degree=3)), ["carat"]),
            ("t", FunctionTransformer(thresholds), ["carat"]),
            ("q", onehot(), QUAL),
            ("ix", make_pipeline(
                ColumnTransformer([("w", FunctionTransformer(np.log), ["carat"]),
                                   ("q", OneHotEncoder(categories=[CATS["color"], CATS["clarity"]], drop="first"),
                                    ["color", "clarity"])]),
                PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
             ["carat", "color", "clarity"])]),
        LinearRegression())),
    "M5 градієнтний бустинг (додатково)": log_target(make_pipeline(
        ColumnTransformer([("w", "passthrough", ["carat"]),
                           ("q", OrdinalEncoder(categories=[CATS[c] for c in QUAL]), QUAL)]),
        HistGradientBoostingRegressor(random_state=SEED))),
}

## Орієнтир: константний прогноз

M0 завжди прогнозує середню ціну навчальної частини, `{python} usd(y_tr.mean())`. Це точність
«без жодних предикторів»: будь-яка корисна модель має бути помітно кращою (результати в @tbl-cv).

## Базова регресія

Спершу найпростіша лінійна модель у доларах (M1):
$price = \beta_0 + \beta_1\,carat + \text{cut} + \text{color} + \text{clarity} + \varepsilon$.

In [ ]:
#| label: fig-m1-resid
#| fig-cap: "M1 (ціна в $): залишки проти прогнозів. Видно дугу (нелінійність), «лійку» (розкид зростає з ціною) і від'ємні прогнози для дрібних каменів."
m1 = smf.ols("price ~ carat + C(cut) + C(color) + C(clarity)", data=train).fit()
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.scatter(m1.fittedvalues, m1.resid, s=2, alpha=0.1)
ax.axhline(0, color="red")
ax.set(xlabel="Прогноз, $", ylabel="Залишок, $", title="M1: залишки проти прогнозів")
plt.tight_layout(); savefig("03_m1_residuals"); plt.show()
KEY["m1_neg"] = int((m1.fittedvalues < 0).sum())

M1 має три вади: `{python} KEY["m1_neg"]` каменів отримують **від'ємну** ціну, залишки утворюють дугу
(зв'язок не лінійний), а розкид росте з ціною. Тому як **базову регресію для висновків** беру
лог-лінійну модель, яку підказали EDA й література (M2):

$$\ln(price) = \beta_0 + \beta_1 \ln(carat) + \sum \gamma_{cut} + \sum \delta_{color} + \sum \theta_{clarity} + \varepsilon$$

Еталонні рівні: огранка Fair, колір J, чистота I1. Коефіцієнти оцінено на всій навчальній частині
OLS зі стійкими до гетероскедастичності похибками HC3.

In [ ]:
#| label: tbl-m2
#| tbl-cap: "M2: ключові коефіцієнти, 95 % ДІ (HC3) і надбавка до ціни за сталих інших змінних"
trc = train.copy()
m2 = smf.ols("log_price ~ log_carat + C(cut) + C(color) + C(clarity)", data=trc).fit(cov_type="HC3")
ci = m2.conf_int()
coef = pd.DataFrame({"β": m2.params, "ДІ 2.5 %": ci[0], "ДІ 97.5 %": ci[1]})
coef["надбавка, %"] = (np.exp(coef["β"]) - 1) * 100
coef["ДІ надбавки, %"] = [f"{(np.exp(a)-1)*100:.0f} … {(np.exp(b)-1)*100:.0f}" for a, b in zip(ci[0], ci[1])]
coef.loc["log_carat", ["надбавка, %", "ДІ надбавки, %"]] = [np.nan, "еластичність"]
coef.loc["Intercept", ["надбавка, %", "ДІ надбавки, %"]] = [np.nan, "ln ціни базового каменя 1 ct"]
coef.index = [i.replace("C(", "").replace(")[T.", " = ").replace("]", "") for i in coef.index]
coef.round(3).to_csv("results/m2_coefficients.csv")
b1, b1_lo, b1_hi = m2.params["log_carat"], ci.loc["log_carat", 0], ci.loc["log_carat", 1]
cl = m2.params.filter(like="clarity")
cl_lo, cl_hi = ci.loc[cl.index, 0], ci.loc[cl.index, 1]
KEY.update(b1=rnd(b1, 3), b1_lo=rnd(b1_lo, 3), b1_hi=rnd(b1_hi, 3),
           r2_m2=rnd(m2.rsquared, 4),
           if_prem=rnd((np.exp(cl.iloc[-1]) - 1) * 100),
           if_lo=rnd((np.exp(cl_lo.iloc[-1]) - 1) * 100), if_hi=rnd((np.exp(cl_hi.iloc[-1]) - 1) * 100),
           si2_prem=rnd((np.exp(cl.iloc[0]) - 1) * 100),
           clar_monotone=bool(np.all(np.diff(np.r_[0, cl.values]) > 0)),
           d_prem=rnd((np.exp(m2.params.filter(like="color").iloc[-1]) - 1) * 100),
           ideal_prem=rnd((np.exp(m2.params.filter(like="cut").iloc[-1]) - 1) * 100))
coef.drop(columns="надбавка, %").round(3)

**Інтерпретація в одиницях задачі** ($R^2$ на шкалі логарифма = `{python} KEY["r2_m2"]`):

- **Вага:** $\hat\beta_1 =$ `{python} KEY["b1"]` (95 % ДІ `{python} KEY["b1_lo"]`–`{python} KEY["b1_hi"]`).
  Камінь на 1 % важчий коштує в середньому на ≈ `{python} rnd(KEY["b1"], 2)` % дорожче. Камінь, удвічі
  важчий за ту саму якість, коштує в $2^{\hat\beta_1} \approx$ `{python} rnd(2 ** b1, 1)` раза більше.
- **Чистота:** порівняно з I1 надбавка зростає від `{python} KEY["si2_prem"]` % (SI2) до
  `{python} KEY["if_prem"]` % (IF, 95 % ДІ `{python} KEY["if_lo"]`–`{python} KEY["if_hi"]` %).
- **Колір:** найкращий колір D дорожчий за J на ≈ `{python} KEY["d_prem"]` %.
- **Огранка:** Ideal дорожча за Fair лише на ≈ `{python} KEY["ideal_prem"]` %. Огранка важить найменше з 4C.

### Діагностика залишків M2

In [ ]:
#| label: fig-m2-diag
#| fig-cap: "Діагностика M2. (а) Залишки проти прогнозів: дуги й лійки M1 майже немає. (б) Середній залишок за вагою: систематична хвиля від −0.12 до +0.11 зі зламами біля 0.5, 1, 1.5 і 2 ct. (в) Q–Q графік: правий хвіст важчий за нормальний."
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
ax[0].scatter(m2.fittedvalues, m2.resid, s=2, alpha=0.08)
ax[0].axhline(0, color="red")
ax[0].set(xlabel="Прогноз ln(ціна)", ylabel="Залишок", title="(а) Залишки проти прогнозів")
bins = np.round(np.arange(0.2, 2.31, 0.05), 2)
grp = pd.cut(train.carat, bins)
rb = m2.resid.groupby(grp, observed=True).agg(["mean", "size"])
rb = rb[rb["size"] >= 30]
ax[1].plot([iv.mid for iv in rb.index], rb["mean"], marker="o", ms=3)
ax[1].axhline(0, color="red")
for t in (0.5, 1, 1.5, 2):
    ax[1].axvline(t, color="gray", ls="--", lw=0.8)
ax[1].set(xlabel="Вага, ct", ylabel="Середній залишок (ln)", title="(б) Середній залишок за вагою")
import scipy.stats as st
st.probplot(m2.resid.sample(4000, random_state=SEED), dist="norm", plot=ax[2])
ax[2].set_title("(в) Q–Q графік залишків (4000 точок)")
plt.tight_layout(); savefig("04_m2_diagnostics"); plt.show()

In [ ]:
#| label: influence
m2_plain = smf.ols("log_price ~ log_carat + C(cut) + C(color) + C(clarity)", data=trc).fit()
bp1 = het_breuschpagan(m1.resid, m1.model.exog)[1]
bp2 = het_breuschpagan(m2_plain.resid, m2_plain.model.exog)[1]
infl = OLSInfluence(m2_plain)
cook = infl.cooks_distance[0]
n_cook = int((cook > 4 / len(trc)).sum())
top = np.argsort(cook)[-int(0.01 * len(trc)):]
m2_wo = smf.ols("log_price ~ log_carat + C(cut) + C(color) + C(clarity)",
                data=trc.drop(trc.index[top])).fit()
sd_ratio = m1.resid[m1.fittedvalues > 10000].std() / m1.resid[m1.fittedvalues < 2000].std()
KEY.update(n_cook=n_cook, cook_max=rnd(float(cook.max()), 4),
           b1_wo=rnd(m2_wo.params["log_carat"], 3), sd_ratio=rnd(sd_ratio, 1))
pd.DataFrame({
    "перевірка": ["Breusch–Pagan, p (M1)", "Breusch–Pagan, p (M2)",
                  "Кук D > 4/n, кількість", "Максимальна D Кука",
                  "β ln(carat) без 1 % найвпливовіших"],
    "значення": [f"{bp1:.1e}", f"{bp2:.1e}", n_cook, rnd(float(cook.max()), 4), KEY["b1_wo"]]})

**Що показала діагностика і що з цього випливає:**

- **Нелінійність.** Логарифми прибрали велику дугу M1, але лишилась систематична «хвиля» за вагою
  (@fig-m2-diag б). Камені 0.4–0.5 ct і 0.55–0.85 ct модель переоцінює (середній залишок до −0.12,
  тобто прогноз на ≈ 11 % завищений), а камені близько 1 ct і найдрібніші недооцінює (до +0.11). Злами
  кривої припадають на 0.5, 1, 1.5 і 2 ct, тобто на «круглі» ваги [@scott2010]. *Наслідок для прогнозу:*
  систематична похибка до ≈ 10 % для цілих груп каменів. *Наслідок для коефіцієнтів:* $\beta_1$ є усередненою
  еластичністю, а не сталою для всіх ваг.
- **Неоднаковий розкид.** У M1 розкид залишків для дорогих каменів у `{python} KEY["sd_ratio"]` раза
  більший, ніж для дешевих. У M2 він значно рівніший, проте тест Бройша–Пагана все одно відхиляє
  гомоскедастичність (при n ≈ 43 000 він чутливий навіть до малих відхилень). Тому для ДІ використано
  **HC3**. Q–Q графік показує важкі хвости, тож прогнозні інтервали на основі нормальності були б
  завузькими.
- **Впливові спостереження.** `{python} KEY["n_cook"]` точок мають D Кука > 4/n, але максимум лише
  `{python} KEY["cook_max"]` (далеко від 1). Без 1 % найвпливовіших точок
  $\hat\beta_1$ = `{python} KEY["b1_wo"]` замість `{python} KEY["b1"]`. Висновки від них не залежать.

## Спроба покращення

**Проблема:** хвиля в залишках за вагою (@fig-m2-diag б). **Модифікації:**

- **M3** замінює $\ln(carat)$ на кубічний сплайн від $\ln(carat)$ (6 вузлів) і додає 4 порогові змінні
  $[carat \ge 0.5], [\ge 1], [\ge 1.5], [\ge 2]$ з роботи Scott і Yelowitz.
- **M4** додатково дозволяє надбавкам за колір і чистоту залежати від ваги (взаємодії
  $\ln(carat)\times$color і $\ln(carat)\times$clarity). У лабораторній №2 різниця в ціні між рівнями
  чистоти була більшою серед великих каменів.

Ridge не розглядаю: предикторів лише ~20 на ~43 000 спостережень, мультиколінеарності між 4C немає,
тож штраф майже не змінить коефіцієнтів.

## Додаткова модель

**M5** — градієнтний бустинг (`HistGradientBoostingRegressor`, параметри за замовчуванням) на тих самих
4 предикторах і тій самій цілі $\ln(price)$. Він сам знаходить пороги й взаємодії, але не має
коефіцієнтів для інтерпретації.

## Порівняння за перехресною валідацією

In [ ]:
#| label: tbl-cv
#| tbl-cap: "5-кратна перехресна валідація на навчальній частині (однакові розбиття). Метрики в доларах; середнє ± стандартне відхилення за 5 розбиттями."
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
rows, folds = [], {}
for name, model in MODELS.items():
    r = cross_validate(model, X_tr, y_tr, cv=kf,
                       scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"])
    mae, rmse, r2 = -r["test_neg_mean_absolute_error"], -r["test_neg_root_mean_squared_error"], r["test_r2"]
    folds[name] = mae
    rows.append({"модель": name, "MAE, $": f"{mae.mean():.0f} ± {mae.std():.0f}",
                 "RMSE, $": f"{rmse.mean():.0f} ± {rmse.std():.0f}",
                 "R²": f"{r2.mean():.3f} ± {r2.std():.3f}",
                 "_mae": mae.mean(), "_sd": mae.std(), "_rmse": rmse.mean()})
cv = pd.DataFrame(rows)
cv.drop(columns=["_mae", "_sd", "_rmse"]).to_csv("results/cv_comparison.csv", index=False)
cv.drop(columns=["_mae", "_sd", "_rmse"])

In [ ]:
#| label: choice
order = cv.sort_values("_mae").reset_index(drop=True)
best = order.loc[0]
chosen = best["модель"]
# простіша модель, якщо гірша менше ніж на 1 SD
simpler = [m for m in MODELS if m != chosen and list(MODELS).index(m) < list(MODELS).index(chosen)
           and "M0" not in m]
for m in simpler:
    if cv.set_index("модель").loc[m, "_mae"] - best["_mae"] < best["_sd"]:
        chosen = m
        break
lin = [m for m in order["модель"] if m[:2] in ("M1", "M2", "M3", "M4")][0]
cvi = cv.set_index("модель")
KEY.update(chosen=chosen, best_linear=lin,
           cv=cv[["модель", "_mae", "_sd", "_rmse"]].round(1).to_dict("records"))
gain_m3 = (1 - cvi.loc["M3 log–log + сплайн і пороги ваги", "_mae"] / cvi.loc["M2 лінійна log–log", "_mae"]) * 100
gain_m4 = (1 - cvi.loc["M4 = M3 + взаємодії вага×колір, вага×чистота", "_mae"]
           / cvi.loc["M3 log–log + сплайн і пороги ваги", "_mae"]) * 100
gain_gb = (1 - cvi.loc["M5 градієнтний бустинг (додатково)", "_mae"] / cvi.loc[lin, "_mae"]) * 100
KEY.update(gain_m3=rnd(gain_m3, 1), gain_m4=rnd(gain_m4, 1), gain_gb=rnd(gain_gb, 1))
Markdown(f"**Обрана модель (до відкриття тесту): {chosen}.** Найкраща лінійна модель: {lin}.")

**Що показала валідація:**

- Усі регресії значно кращі за орієнтир M0. Перехід до логарифмів (M1 → M2) зменшує MAE приблизно
  вдвічі. Отже, форма зв'язку важить більше за вибір предикторів.
- **Сплайн і пороги (M3)** зменшують MAE ще на `{python} KEY["gain_m3"]` % порівняно з M2, різниця
  набагато більша за розкид між розбиттями. Модифікація розв'язала проблему, яку показала діагностика.
- **Взаємодії (M4)** зменшують MAE ще на `{python} KEY["gain_m4"]` % порівняно з M3.
- **Бустинг (M5)** точніший за найкращу лінійну модель на `{python} KEY["gain_gb"]` %, але не має
  коефіцієнтів. Тому гіпотези перевіряю в M2, а для прогнозу беру модель за правилом вибору.

## Фінальна оцінка на тесті

In [ ]:
#| label: tbl-test
#| tbl-cap: "Одноразова оцінка на тестовій частині. Моделі перенавчено на всій навчальній частині."
final = {}
for name in dict.fromkeys(["M0 константа (середнє)", chosen, lin]):
    m = MODELS[name].fit(X_tr, y_tr)
    p = m.predict(X_te)
    final[name] = p
test_rows = [{"модель": n, "MAE, $": rnd(mean_absolute_error(y_te, p)),
              "RMSE, $": rnd(root_mean_squared_error(y_te, p)),
              "R²": rnd(r2_score(y_te, p), 3),
              "медіана |помилки| / ціна, %": rnd(np.median(np.abs(y_te - p) / y_te) * 100, 1)}
             for n, p in final.items()]
tt = pd.DataFrame(test_rows)
tt.to_csv("results/test_metrics.csv", index=False)
KEY["test"] = tt.to_dict("records")
tt

In [ ]:
#| label: fig-test
#| fig-cap: "Тест: фактична ціна проти прогнозу обраної моделі (log-шкали; червона лінія — ідеальний прогноз) і абсолютна помилка за групами ваги."
pred = final[chosen]
err = test.assign(pred=pred, abs_err=np.abs(test.price - pred),
                  rel_err=np.abs(test.price - pred) / test.price * 100)
err["група ваги"] = pd.cut(err.carat, [0, 0.5, 0.75, 1, 1.5, 2, 5.1],
                           labels=["до 0.5", "0.5–0.75", "0.75–1", "1–1.5", "1.5–2", "понад 2"])
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
ax[0].scatter(err.price, err.pred, s=3, alpha=0.15)
lim = [300, 20000]
ax[0].plot(lim, lim, color="red")
ax[0].set(xscale="log", yscale="log", xlim=lim, ylim=lim, xlabel="Фактична ціна, $",
          ylabel="Прогноз, $", title=f"Фактичне і прогноз: {chosen.split(' ')[0]}")
sns.boxplot(data=err, x="група ваги", y="abs_err", ax=ax[1], showfliers=False, color="lightblue")
ax[1].set(xlabel="Вага, ct", ylabel="|помилка|, $", title="Абсолютна помилка за вагою")
plt.tight_layout(); savefig("05_test_pred"); plt.show()
by = err.groupby("група ваги", observed=True).agg(
    n=("price", "size"), median_price=("price", "median"),
    MAE=("abs_err", "mean"), median_rel_err=("rel_err", "median")).round(1)
by.to_csv("results/test_errors_by_carat.csv")
KEY["by_carat"] = by.reset_index().astype({"група ваги": str}).to_dict("records")
by.rename(columns={"n": "каменів", "median_price": "медіана ціни, $",
                   "MAE": "MAE, $", "median_rel_err": "медіана відн. помилки, %"})

In [ ]:
#| label: worst
cap = err[err.price > 15000]
KEY.update(mae_cap=rnd(cap.abs_err.mean()), n_cap=len(cap),
           share_big_err=rnd(err.loc[err.carat >= 1.5, "abs_err"].sum() / err.abs_err.sum() * 100),
           share_big_n=rnd((err.carat >= 1.5).mean() * 100))
worst = err.sort_values("abs_err", ascending=False).head(6)
worst[["carat", "cut", "color", "clarity", "price", "pred", "abs_err"]].round(0)

**Де помилки найбільші.** У доларах помилка росте з вагою й ціною: камені від 1.5 ct становлять
`{python} KEY["share_big_n"]` % тесту, але дають `{python} KEY["share_big_err"]` % сумарної абсолютної
помилки. У відсотках картина інша: медіанна відносна помилка ≈ 5 % для каменів 0.5–2 ct, але
`{python} KEY["by_carat"][0]["median_rel_err"]` % для дрібних (до 0.5 ct) і
`{python} KEY["by_carat"][-1]["median_rel_err"]` % для каменів понад 2 ct. Розрив на графіку біля 1 500 $
походить з даних: у наборі майже немає цін між 1 455 і 1 545 $ (див. лабораторну №2).
Найбільші абсолютні помилки — у дорогих каменів поблизу верхньої межі цін набору (`{python} KEY["n_cap"]` каменів дорожчих за 15 000 $, MAE `{python} usd(KEY["mae_cap"])`):
там ціни обрізані, а інші характеристики (флуоресценція, пропорції, сертифікат, продавець) у дані не
потрапили.

In [ ]:
#| label: save-key
with open("results/key_numbers.json", "w", encoding="utf-8") as f:
    json.dump(KEY, f, ensure_ascii=False, indent=2, default=str)

# Обговорення й висновки

**Відповідь на дослідницьке питання.** Чотири характеристики з сертифіката прогнозують ціну нового
діаманта з того самого ринку дуже добре. Обрана модель на тесті помиляється в середньому на
`{python} usd(KEY["test"][1]["MAE, $"])`, тоді як константний прогноз — на
`{python} usd(KEY["test"][0]["MAE, $"])`. Типова (медіанна) відносна помилка становить
`{python} KEY["test"][1]["медіана |помилки| / ціна, %"]` %. Найкраща лінійна модель трохи гірша
(MAE `{python} usd(KEY["test"][-1]["MAE, $"])`), зате пояснює, як саме 4C пов'язані з ціною.

**H1 (вага).** Еластичність `{python} KEY["b1"]`, 95 % ДІ `{python} KEY["b1_lo"]`–`{python} KEY["b1_hi"]`.
Інтервал вузький і цілком лежить вище 1, тож за сталих інших C ціна зростає з вагою швидше, ніж
пропорційно. Дані **узгоджуються з H1**. Оцінка більша за нахил 1.68 з лабораторної №2, бо там вага
не була «очищена» від якості: великі камені частіше мають гіршу чистоту й колір. Застереження:
діагностика показала, що еластичність не стала (стрибки на круглих вагах), а гіпотеза походить з EDA,
тож результат розвідувальний.

**H2 (чистота).** Усі 7 коефіцієнтів чистоти додатні, з вузькими ДІ, і зростають від SI2 до IF
(`{python} "монотонно" if KEY["clar_monotone"] else "не зовсім монотонно"`). Надбавка IF над I1 становить
`{python} KEY["if_prem"]` % (95 % ДІ `{python} KEY["if_lo"]`–`{python} KEY["if_hi"]` %), що більше за
100 %. Дані **узгоджуються з H2** (так само розвідувально).

**Якість прогнозу на нових даних.** Різниця між MAE на перехресній валідації й на тесті мала, отже модель
не перенавчена. Однак тест не повністю незалежний: рішення про логарифми й набір предикторів ухвалені
після EDA всього набору.

**Межі узагальнення й змішувальні чинники.**

- Дані — один ринок і один рік (2008), лише круглі камені, ціни до 18 823 $. На інші роки, валюти,
  форми каменів чи синтетичні діаманти результат не переноситься.
- Це **спостережні** дані, тому коефіцієнти описують **асоціації** в каталогах, а не причинний вплив.
  Змішувальні чинники, яких немає в даних: продавець і його націнка, лабораторія сертифікації (GIA чи інша),
  флуоресценція, симетрія й полірування, дата пропозиції.
- Ціни взято з каталогу (ціни пропозиції), а не з реальних угод.

**Продовження дослідження.** Зібрати свіжі дані з кількох онлайн-магазинів з датою, продавцем,
лабораторією сертифікації й флуоресценцією. Перевірити зафіксовані моделі (M5 для прогнозу, M4 для
інтерпретації) на них як на справді
незалежному тесті та оцінити, чи збереглися надбавки за «круглі» ваги через 15+ років.

# Бібліографія {.unnumbered}

::: {#refs}
:::